## TP1 - Estimation de $\lambda$ et $\mu$ sur des architectures

**Travail pratique de modélisation des files d'attente**

## Objectifs pédagogiques
- Identifier ce qu'est un « client » et un « serveur » dans une architecture informatique réelle (web, mobile).
- Estimer le taux d'arrivée $\lambda$ et le taux de service $\mu$ par les méthodes du moment et du maximum de vraisemblance.

## Pré-requis
- Connaissance des modèles M/M/1 et M/M/c, de la notation de Kendall et de la loi de Little.
- Maîtrise élémentaire de Python et de Pandas (lecture CSV, agrégation, tracé).
- Notions de statistique descriptive (moyenne, variance, histogramme).

## Livrables
- Un notebook Jupyter ou un script Python complet et commenté.*italicized text*

## Partie 0 — Mise en place et rappels

### 0.1 Le triplet client - serveur - discipline dans une architecture
Avant tout calcul, il faut identifier dans chaque système ce qui constitue un client, ce qui constitue un serveur, et ce qui se trouve dans la file d'attente.

| Architecture | Client | Serveur(s) | File d'attente |
| :--- | :--- | :--- | :--- |
| Serveur web | Requête HTTP | Worker / process / thread | Backlog TCP, queue applicative |
| API mobile | Appel REST | Pool de threads servlet | Queue du pool de threads
### 0.2 Ce que mesure un log
Les serveurs produisent presque tous des logs au format CSV ou texte structuré.
Pour estimer $\lambda$ et $\mu$ il vous faut deux colonnes au minimum :
- **Timestamp d'arrivée** : l'instant où la requête a touché le système.
- **Durée de traitement** : le temps total entre l'entrée et la sortie de la requête.

### 0.3 Méthode d'estimation
Pour un échantillon de $n$ inter-arrivées supposées indépendantes et exponentielles de paramètre $\lambda$, l'estimateur du maximum de vraisemblance est :
$$\hat{\lambda} = \frac{1}{\bar{T}} \quad \text{où} \quad \bar{T} = \frac{1}{n} \sum T_i$$

Et de même pour les durées de service $S_i$ supposées exponentielles de paramètre $\mu$ :
$$\hat{\mu} = \frac{1}{\bar{S}}$$
Soit un échantillon de variables aléatoires :
$T_1, T_2, \dots, T_n$
 **indépendantes et identiquement distribuées** suivant une loi exponentielle de paramètre $ \lambda > 0 $ (voir cours) :

$$
T_i \sim \text{Exp}(\lambda)
$$

La densité de probabilité est :
$$
f(t; \lambda) = \lambda e^{-\lambda t}, \quad t \ge 0
$$

---
La fonction de vraisemblance est donnée par :
$$ \mathcal{L}(\lambda) = \prod_{i=1}^{n} f(T_i; \lambda) $$

Donc :
$$
\mathcal{L}(\lambda) = \prod_{i=1}^{n} \lambda e^{-\lambda T_i}
= \lambda^n \exp\left(-\lambda \sum_{i=1}^{n} T_i \right)
$$

---

Avec la log-vraisemblance :

$$
\ell(\lambda) = \log \mathcal{L}(\lambda)
$$

$$
\ell(\lambda) = n \log \lambda - \lambda \sum_{i=1}^{n} T_i
$$

---

On dérive par rapport à \(\lambda\) :

$$
\frac{d\ell}{d\lambda} = \frac{n}{\lambda} - \sum_{i=1}^{n} T_i
$$

On cherche le point critique :

$$
\frac{n}{\lambda} - \sum T_i = 0
$$

---
On résout :

$$
\frac{n}{\lambda} = \sum_{i=1}^{n} T_i
$$

$$
\hat{\lambda} = \frac{n}{\sum_{i=1}^{n} T_i}
$$

En introduisant la moyenne empirique :

$$
\bar{T} = \frac{1}{n} \sum_{i=1}^{n} T_i
$$

on obtient :

$$
\hat{\lambda} = \frac{1}{\bar{T}}
$$

Si
$
S_1, \dots, S_n \sim \text{Exp}(\mu)
$

Alors, de manière identique :

$$
\hat{\mu} = \frac{1}{\bar{S}}
$$

avec :
$$
\bar{S} = \frac{1}{n} \sum_{i=1}^{n} S_i
$$

---

## Interprétation

- $ \lambda $ : taux d'arrivée (nombre moyen d'événements par unité de temps)
- $ \mu $ : taux de service

Ainsi :
$$
\lambda = \frac{1}{\mathbb{E}[T]}, \quad \mu = \frac{1}{\mathbb{E}[S]}
$$

Les estimateurs remplacent simplement l'espérance par la moyenne empirique.

## Partie 1 — Serveur web (Nginx)

### 1.1 Contexte
Vous administrez le serveur Nginx d'une application. Vous voulez savoir si le serveur actuel est correctement dimensionné.
Vous disposez d'un extrait du fichier `access.log` enrichi avec `request_time`.

### 1. Questions

#### Question 1.1 — Lecture et exploration
Charger le fichier `nginx_access.csv`. Convertir la colonne timestamp en datetime.
Afficher le nombre total de requêtes, la plage horaire couverte et la statistique descriptive de `request_time`.

#### Question 1.2 — Estimation de $\lambda$
Calculer la suite des inter-arrivées (différences entre timestamps successifs). En déduire l'estimateur $\hat{\lambda}$ par la moyenne empirique.
Donner la valeur en req/s et req/min.

#### Question 1.3 — Estimation de $\mu$
À partir de la colonne `request_time`, calculer l'estimateur $\hat{\mu}$. Comparer la moyenne et l'écart-type pour discuter de l'hypothèse exponentielle.

#### Question 1.4 — Vérification graphique des hypothèses
Tracer l'histogramme des inter-arrivées et superposer la densité exponentielle ajustée. Faire de même pour les durées de service.

## Partie 2 — API mobile (M/M/c)

### 2.1 Contexte
L'application mobile consomme une API REST hébergée sur un serveur Spring Boot
avec un pool de threads servlet. Le fichier `api_mobile.csv` contient les colonnes
`arrival`, `service`, `start`, `departure`. La taille du pool **n'est pas
documentée** : il faut la retrouver à partir des données.

### 2.2 — Estimation de λ et μ
Charger `api_mobile.csv`. Calculer les inter-arrivées et estimer
$\hat{\lambda} = 1/\bar{T}$, puis estimer $\hat{\mu} = 1/\bar{S}$ à partir de la
colonne `service`. Reporter les coefficients de variation des inter-arrivées
et des durées de service ; conclure sur la plausibilité des hypothèses
markoviennes.

### 2.3 — Estimation empirique du nombre de serveurs c

Le fichier contient une colonne `thread` indiquant l'identifiant du thread ayant traité chaque requête.

**Principe.** À tout instant t, le nombre de requêtes en cours de service est :
$$N_s(t) = \#\{ i \;:\; \texttt{start}_i \le t < \texttt{departure}_i \}$$
La capacité c du pool est, par définition, le **maximum** que cette quantité peut
atteindre. Si le système est suffisamment chargé pendant la période d'observation,
on aura presque sûrement $\max_t N_s(t) = c$.

**Question 2.3.a.** Implémenter la fonction `serveurs_occupes(log, t)` qui renvoie
$N_s(t)$. L'évaluer sur une grille fine d'instants couvrant la durée de simulation
et tracer $N_s(t)$ en fonction du temps.

**Question 2.3.b.** En déduire l'estimateur :
$$\hat{c} = \max_t N_s(t)$$
Donner sa valeur. Tracer une ligne horizontale rouge à $\hat{c}$ sur le graphique
précédent.

**Question 2.3.c.** Calculer la **fraction du temps** où le pool est saturé,
c'est-à-dire la proportion d'instants où $N_s(t) = \hat{c}$. Interpréter : un
système souvent saturé est sous-dimensionné.
**Question 2.3.c.** Calculer le taux d'utilisation global du système $$ \rho $$  à l'aide de la formule :

$$\rho = \frac{\lambda}{c\mu}$$

Interpréter la valeur obtenue afin d’évaluer la stabilité du modèle avant de procéder à l’analyse empirique.